# Sensitivity analysis

Companion to Notebook 5. Every mechanism-result track rests on a handful of
explicit, named threshold/percentile choices -- this notebook re-examines
each of those choices against the real fleet data, using
`src/dnsp_analysis/sensitivity.py`.

| Stage | Choice being tested | Cost | Writes anything? |
|---|---|---|---|
| 1 | DER-phase mapping confidence thresholds | free | no |
| 2 | Capacity-proxy percentile | moderate | yes -- new namespaced `capacity_proxy_*.parquet` files only, never the production p99 build |
| 3 | Volt-VAr Q_impact bucket boundaries | free | no |
| 4 | Volt-VAr tolerance fraction | moderate | no |
| 5a | Site-level conformance threshold | free (inline) | no |
| 5b | Power-coverage eligibility gate cutoff | free (inline) | no |

"Free" means the check reuses an already-cached, already-built
intermediate (a cached pandas frame or a single DuckDB histogram query) and
costs nothing more per extra candidate value. "Moderate" means each
candidate value needs its own fresh DuckDB query, but never a full
per-serial/month/voltage-bin mechanism-result rebuild -- see
`sensitivity.py`'s module docstring for the exact reasoning per function.

No function here ever silently picks a "better" threshold and rebuilds
production output with it -- every result below is read-only exploration to
inform a deliberate, separately-recorded decision (an `analysis.toml`
edit + a Notebook 4/5 rebuild), exactly like every other proxy/threshold
choice in this project.

In [ ]:
from __future__ import annotations

import dataclasses
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'src' / 'dnsp_analysis').is_dir()), None)
assert PROJECT_ROOT is not None, 'Start Jupyter inside the dnsp_analysis project.'
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from dnsp_analysis.as4777_curves import Q_IMPACT_THRESHOLDS
from dnsp_analysis.analysis_cohort import site_eligibility_path
from dnsp_analysis.config import load_config
from dnsp_analysis.mechanism_config import load_mechanism_config
from dnsp_analysis import result_views as rv
from dnsp_analysis import sensitivity as sn

CONFIG_PATH = PROJECT_ROOT / 'analysis.toml'
config = load_config(CONFIG_PATH, check_inputs=True)
mechanism_der_inferred = load_mechanism_config(CONFIG_PATH)
mechanism_p99_proxy = dataclasses.replace(mechanism_der_inferred, capacity_basis='p99_net_export_proxy').validate()

pd.set_option('display.max_columns', 100)
plt.style.use('seaborn-v0_8-whitegrid')

## Stage 0: scope

`mechanism_p99_proxy` is used throughout -- it is the one track with real
`n_assessable > 0` today (`s_rated_kva` is null fleet-wide, so
`mechanism_der_inferred`'s own Volt-VAr denominator is 100% unassessable;
see Notebook 5 Stage 3). Sensitivity results below describe how *that*
track's classification would shift under alternative thresholds, not a
claim about the verified `s_rated_kva` basis (which stays unassessable
regardless of any threshold tested here).

In [ ]:
SAMPLE_MONTH = '2025-04'
SAMPLE_SITE_BUCKET = 0
sample_scope = config.scope(SAMPLE_MONTH, SAMPLE_SITE_BUCKET)
full_scope = config.scope(None, None)

print('Q_IMPACT_THRESHOLDS (production):', Q_IMPACT_THRESHOLDS)
print('tolerance_fraction (production):', mechanism_p99_proxy.tolerance_fraction)
print('capacity_proxy_percentile (production):', mechanism_p99_proxy.capacity_proxy_percentile)

## Stage 1: DER-phase mapping confidence thresholds (free)

`telemetry_profiles.derive_site_profiles` is pure pandas over the already-
built `site_phase_profile.parquet` -- every variant below costs seconds, not
a requery of raw telemetry. Default sweep perturbs each of
`phase_mapping_min_signature_w`, `phase_mapping_high_margin_ratio`,
`phase_mapping_medium_margin_ratio` independently by +/-25%, holding the
other two at their `analysis.toml` values.

In [ ]:
phase_mapping_sweep = sn.phase_mapping_sensitivity(config, full_scope)
display(phase_mapping_sweep)

fig, ax = plt.subplots(figsize=(10, 4.5))
phase_mapping_sweep.set_index('variant')[['n_high', 'n_medium', 'n_low', 'n_insufficient', 'n_unknown']].plot.barh(
    stacked=True, ax=ax
)
ax.set_xlabel('sites')
ax.set_title('DER-phase mapping confidence under alternative thresholds')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

production_row = phase_mapping_sweep.loc[phase_mapping_sweep['variant'] == 'production'].iloc[0]
print(f"Production: {int(production_row['n_high'])} high, {int(production_row['n_medium'])} medium, "
      f"{int(production_row['n_mapping_assessable'])} mapping-assessable of {int(production_row['n_sites'])} sites.")

## Stage 2: capacity-proxy percentile (moderate)

Each candidate percentile costs one fresh full-table `quantile_cont` scan
(`capacity_proxy.build_capacity_proxy`) -- there is no cheaper cached
intermediate to resample from. Every candidate writes to its own
namespaced path (never overwrites the production p99 build); the production
percentile itself (0.99) is read back from the file Notebook 4 already
built, rather than rebuilt here.

In [ ]:
capacity_sweep = sn.capacity_percentile_sensitivity(
    config, full_scope, mechanism_p99_proxy,
    percentiles=(0.90, 0.95, 0.97, 0.995),
)

import duckdb
from dnsp_analysis.mechanism_paths import capacity_proxy_path
production_path = capacity_proxy_path(config, full_scope, mechanism_p99_proxy)
production_stats = duckdb.connect().execute(
    f"""
    SELECT
        {mechanism_p99_proxy.capacity_proxy_percentile} AS capacity_proxy_percentile,
        count(*) AS sites,
        count_if(capacity_proxy_va IS NULL) AS n_null_proxy,
        min(capacity_proxy_va) AS min_capacity_proxy_va,
        avg(capacity_proxy_va) AS mean_capacity_proxy_va,
        median(capacity_proxy_va) AS median_capacity_proxy_va,
        max(capacity_proxy_va) AS max_capacity_proxy_va,
        {str(production_path)!r} AS output
    FROM read_parquet({str(production_path)!r})
    """
).fetchdf()
production_stats['output'] = production_stats['output'] + '  (production, already built)'

capacity_sweep_full = pd.concat([capacity_sweep, production_stats], ignore_index=True).sort_values('capacity_proxy_percentile')
display(capacity_sweep_full.drop(columns=['output']))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(capacity_sweep_full['capacity_proxy_percentile'], capacity_sweep_full['median_capacity_proxy_va'], marker='o', label='median')
ax.plot(capacity_sweep_full['capacity_proxy_percentile'], capacity_sweep_full['mean_capacity_proxy_va'], marker='o', label='mean')
ax.axvline(mechanism_p99_proxy.capacity_proxy_percentile, color='grey', linestyle='--', label='production')
ax.set_xlabel('capacity_proxy_percentile')
ax.set_ylabel('capacity_proxy_va')
ax.set_title('Empirical capacity proxy vs. percentile choice')
ax.legend()
plt.tight_layout()
plt.show()

## Stage 3: Volt-VAr Q_impact bucket boundaries (free)

The non-conformant assessable population is binned by Q_impact exactly
once, then re-bucketed under any number of candidate threshold sets by
resumming those cached bins -- no additional query per candidate. The
`conformant` bucket is a literal band-membership check, independent of
these thresholds, so its count (and `n_assessable`) is held fixed across
every row below. Default sweep scales the production inactive/
minor_deviation band half-widths by a shared factor
(`0.5` narrower ... `1.5` wider); `band_x1` reproduces `Q_IMPACT_THRESHOLDS`
exactly.

In [ ]:
q_impact_sweep = sn.q_impact_bucket_sensitivity(config, full_scope, mechanism_p99_proxy)
display(q_impact_sweep)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(q_impact_sweep['threshold_set'], q_impact_sweep['conformance_fraction'], marker='o', label='conformance_fraction')
ax.plot(q_impact_sweep['threshold_set'], q_impact_sweep['non_conformance_fraction'], marker='o', label='non_conformance_fraction')
ax.set_ylabel('fraction of n_assessable')
ax.set_title('Conformance rollup under alternative Q_impact bucket boundaries')
ax.legend()
plt.tight_layout()
plt.show()

## Stage 4: Volt-VAr tolerance fraction (moderate)

`tolerance_fraction` changes the required band itself (both the
`conformant` literal check and Q_impact's own denominator shift), so
Stage 3's cached histogram cannot answer this question -- each candidate
value needs its own fresh, fleet-aggregated query. Still far cheaper than a
full `build_voltvar_results` rebuild (no per-serial/month/voltage-bin
breakdown).

In [ ]:
tolerance_sweep = sn.tolerance_fraction_sensitivity(
    config, full_scope, mechanism_p99_proxy,
    tolerance_fractions=(0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.10),
)
display(tolerance_sweep)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(tolerance_sweep['tolerance_fraction'], tolerance_sweep['conformance_fraction'], marker='o')
ax.axvline(mechanism_p99_proxy.tolerance_fraction, color='grey', linestyle='--', label='production')
ax.set_xlabel('tolerance_fraction')
ax.set_ylabel('conformance_fraction')
ax.set_title('Conformance rollup vs. tolerance fraction')
ax.legend()
plt.tight_layout()
plt.show()

## Stage 5: two checks needing no new library code (free, inline)

Both of these re-threshold an already-persisted continuous value that
`result_views.py`/`analysis_cohort.py` already compute -- no fresh query,
no new function in `sensitivity.py`.

In [ ]:
# 5a. Site-level conformance threshold (voltvar_site_conformance_view already
# takes conformance_threshold as a parameter -- just call it repeatedly).
conformance_threshold_sweep = {}
for threshold in (0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
    frame = rv.voltvar_site_conformance_view(
        config, full_scope, mechanism=mechanism_p99_proxy, conformance_threshold=threshold
    )
    conformance_threshold_sweep[threshold] = frame['site_status'].value_counts()
conformance_threshold_sweep = pd.DataFrame(conformance_threshold_sweep).fillna(0).astype(int).T
conformance_threshold_sweep.index.name = 'conformance_threshold'
display(conformance_threshold_sweep)

In [ ]:
# 5b. Power-coverage eligibility gate cutoff -- re-threshold
# minimum_joint_power_coverage from site_eligibility.parquet directly,
# holding every other gate fixed at its persisted value.
eligibility = pd.read_parquet(site_eligibility_path(config))
other_gates = (
    eligibility['gate_solar_only']
    & eligibility['gate_no_battery']
    & eligibility['gate_no_controlled_load']
    & eligibility['gate_mapping']
)
power_coverage_sweep = []
for cutoff in (0.80, 0.85, 0.90, 0.95, 0.98, 1.00):
    passes_all = other_gates & eligibility['minimum_joint_power_coverage'].ge(cutoff)
    power_coverage_sweep.append(
        {
            'power_coverage_cutoff': cutoff,
            'sites_passing_power_coverage_gate': int(
                eligibility['minimum_joint_power_coverage'].ge(cutoff).sum()
            ),
            'sites_passing_all_gates': int(passes_all.sum()),
        }
    )
power_coverage_sweep = pd.DataFrame(power_coverage_sweep)
display(power_coverage_sweep)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(power_coverage_sweep['power_coverage_cutoff'], power_coverage_sweep['sites_passing_all_gates'], marker='o')
ax.axvline(0.95, color='grey', linestyle='--', label='production default (CohortRules.minimum_power_coverage)')
ax.set_xlabel('minimum_joint_power_coverage cutoff')
ax.set_ylabel('sites passing every eligibility gate')
ax.set_title('Eligible-site count vs. power-coverage gate cutoff')
ax.legend()
plt.tight_layout()
plt.show()

## Stage 6: interpretation checklist

**Free checks (Stages 1, 3, 5a, 5b):** cost nothing beyond the first query,
so every candidate value shown above is exact (Stage 1, 5a, 5b) or exact up
to a small histogram-bin quantization error (Stage 3, tightened by
`bin_width`). Safe to re-run with a wider or finer sweep at any time.

**Moderate checks (Stages 2, 4):** each candidate value is its own fresh,
fleet-aggregated query -- still far cheaper than a full mechanism-result
rebuild, but not free to extend indefinitely.

**Nothing here changes production output.** Stage 2 is the only cell that
writes anything, and it writes to new, namespaced `capacity_proxy_*.parquet`
paths only -- the production p99 build Notebook 4/5 read is untouched.
Adopting a different threshold/percentile as the new production default is
a separate, deliberate step: edit the relevant `analysis.toml` value, record
why (mirroring the `active_sign_review_state` change note already in that
file), then rebuild via Notebook 4 and re-validate via Notebook 5 -- never
done silently from this notebook.